# L2: Building a Basic Agent UI

In this lesson, you'll wire up a LangChain agent to a React chat interface using CopilotKit and the AG-UI protocol — the same pattern used to connect any agent backend to any frontend.

## 📋 Learning Objectives

1. **Run a LangChain agent** - Start an AG-UI compatible backend with FastAPI
2. **Set up CopilotKit** - Connect a React frontend to your agent via CopilotRuntime
3. **Swap agent backends** - Switch between LangChain/OpenAI and Google ADK/Gemini without changing UI code

## Before you start

This lesson uses `%%writefile` to save code directly to the frontend project on disk. The running app picks up those changes automatically — so you'll see your UI update in real time as you work through the cells.

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Already completed this lesson?</b> If you want to start this notebook over again, run the cell below to reset all files to their original state<b>Skip this if it's your first time through.</b>
</div>

In [ ]:
# from helper import reset_lesson
# reset_lesson(2)

## What you'll build

A chat interface powered by a LangChain agent, connected through CopilotKit. By the end, you'll be able to chat with your agent directly in the browser:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Please write me a poem!</div>

<img src="images/copilotkit-chat.png" alt="CopilotKit Chat" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## Install dependencies

In this learning environment, all backend and frontend dependencies are pre-installed. Run the next two cells to confirm everything is ready.

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Running outside this environment?</b> Install backend dependencies with <code>pip install -r requirements.txt</code>, then run <code>npm install</code> in the <code>frontend/</code> directory.
</div>

In [ ]:
# supress warning messages
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# !pip install -r requirements.txt

In [ ]:
from helper import install_frontend
install_frontend()

Now you will load the API keys for the models your agents will use. In this learning environment, the DeepLearning.AI platform provides these keys, so you only need to load them.
    

In [ ]:
from helper import load_api_keys
load_api_keys()

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Running this on your own computer?</b> You'll need to get your own API keys:

- **OpenAI** (used in the main lesson): [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **Google AI** (used in the bonus section): [aistudio.google.com/apikey](https://aistudio.google.com/app/apikey)
</div>

## Build the agent

### Start the server

CopilotKit connects to your agent through an AG-UI compatible HTTP endpoint. Here, you'll start a FastAPI server and mount a `LangGraphAGUIAgent`.

In [ ]:
from fastapi import FastAPI

# CopilotKit and AG-UI dependencies for the AG-UI server
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent

# Simple helper that starts the server and manages port conflicts
from helper import start_server

# Build the AG-UI endpoint into a FastAPI app
app = FastAPI()
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="lesson2_agent",
    description="Lesson 2 chart agent",
    graph=graph,
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Start the server
start_server(app, port=8002)

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>What is AG-UI?</b> <a href="https://docs.ag-ui.com">AG-UI</a> is an open protocol for connecting agent backends to frontends. CopilotKit uses it under the hood. You'll learn more about how it works at the end of this lesson.
</div>

### Define agent

Here you'll create a LangChain agent with an OpenAI model, a memory checkpointer, and CopilotKit middleware.

Re-run this cell whenever you want to change the agent's configuration — `agent.graph = ...` hot-reloads the graph without restarting the server.

In [ ]:
from copilotkit import CopilotKitMiddleware

# LangChain agent imports
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Make a pre-built LangChain agent
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=("You are a helpful assistant"),
)

# Update the agent's graph (hot reloads)
agent.graph = graph
print("✓ Agent graph updated!")

`CopilotKitMiddleware()` is what connects your agent to CopilotKit — it lets the model discover and call frontend tools, which you'll build in L3.

Without the middleware, the agent only sees backend-defined tools.

## Getting Started with CopilotKit

CopilotKit ships with both fully-headless UI and pre-built React components for agent UIs. In this lesson you'll be using three pieces:

- **`CopilotRuntime`**: A secure bridge that connects your frontend to any agent backend.
- **`CopilotKit`**: React provider that configures the runtime connection for your app.
- **`CopilotChat`**: A batteries-included, customizable chat interface for your agents.

### Start Frontend

You'll be using a pre-built application as the backdrop for the lessons. As you make changes, it will automatically pick up those changes and display them for you.

Run the next two cells to start the dev server and open a live preview.

In [ ]:
from helper import start_frontend
start_frontend(port=3002)

Finally, you can open a live-preview to the running application.

In [ ]:
from helper import display_app
display_app(port=3002)

Next, you will setup a chat to talk with the agent in this application. To start, this will just be an empty chat app. 

### Set up the `CopilotRuntime`

The `CopilotRuntime` is the secure bridge between your frontend and agent backend. Here, you'll register your LangChain agent as the `default` agent:

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Why <code>/v2</code>?</b> CopilotKit packages export both v1 and v2 APIs. The <code>/v2</code> import path gives you the latest hooks and components. All code in this course uses v2.
</div>

### Wrap your application in the `CopilotKit` provider

The `CopilotKit` provider connects your React app to the runtime via `runtimeUrl`. Place it in `main.tsx` to wrap the entire application — or around individual components if you prefer.

In [ ]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

### Set up the `CopilotChat` component

Finally, add the `CopilotChat` component and point it at the `default` agent you registered in the `CopilotRuntime`.

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Note:</b> You can also use CopilotKit in headless mode (without a pre-built chat UI) via the <code>useAgent</code> hook. In this lesson, you use the built-in <code>CopilotChat</code> component for simplicity.
</div>

### Give it a try!

Your app now has a working chat interface. Run the cell below to open a preview and try chatting with your agent.

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Hello chat!</div>

In [ ]:
from helper import display_app
display_app(port=3002)

## Bonus - Connect to Google ADK

In this section, you'll add a second agent backend using Google ADK and Gemini — and swap to it without changing any UI code.

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Note:</b> This section requires a <code>GEMINI_API_KEY</code>. You can get your own Gemini API key from <a href="https://aistudio.google.com/app/apikey" target="_blank">Google AI Studio</a>.
</div>

### Setup an ADK Agent
First, create and start a Google ADK agent on port `8009`:

In [ ]:
from fastapi import FastAPI
from ag_ui_adk import ADKAgent, add_adk_fastapi_endpoint
from google.adk.agents import LlmAgent
from helper import start_server

gemini_agent = LlmAgent(
    name="assistant",
    model="gemini-2.5-flash",
    instruction="Be helpful and fun!",
)

adk_agent = ADKAgent(
    adk_agent=gemini_agent,
    app_name="demo_app",
    user_id="demo_user",
    session_timeout_seconds=3600,
    use_in_memory_services=True,
)

app_adk = FastAPI()
add_adk_fastapi_endpoint(app_adk, adk_agent, path="/")

start_server(app_adk, port=8009)

### Update the `CopilotRuntime` to register the agent
Now update the runtime to register the Gemini agent alongside the default one:

In [ ]:
%%writefile frontend/server.ts

import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { HttpAgent } from "@ag-ui/client";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";
import { serve } from "@hono/node-server";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const adkAgent = new HttpAgent({
  url: process.env.ADK_AGENT_URL || "http://localhost:8009",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
    gemini: adkAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

### Update the frontend to use Gemini

To switch models, update the `agentId` in `frontend/src/App.tsx`:
- `"default"` -> LangChain/OpenAI
- `"gemini"` -> ADK/Gemini

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

export const agentId = "gemini";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

### Display the app

Your chat is now connected to the Gemini backend. Try asking it some questions and notice the differences in behavior.

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">What model are you powered by?</div>

In [ ]:
from helper import display_app
display_app(port=3002)

## What is AG-UI?

AG-UI (Agent-User Interaction) is an open, event-based protocol for connecting agent backends to frontends. It standardizes how chat messages, tool calls, state updates, and streaming tokens flow over HTTP.

You just saw this in action — here's why it matters:
- CopilotKit can talk to any backend that implements AG-UI.
- You swapped from LangChain/OpenAI to ADK/Gemini with a config change — no UI rewrites.
- Streaming and tool behavior stay consistent across frameworks.

<img src="images/protocols.png" alt="AG-UI protocol diagram" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />


## What you learned

- How to run a LangChain agent behind an AG-UI endpoint and connect it to CopilotKit.
- How to hot-reload the agent graph during development.
- How the same frontend can switch between LangChain/OpenAI and ADK/Gemini backends.

## Next step

In **Lesson 3**, you'll focus on **controlled generative UI**:
- Registering typed frontend tools with `useComponent()`
- Rendering structured tool outputs in chat
- Keeping model-driven UI behavior predictable and safe
